# Convertible GAN + Classification Project

This notebook keeps the full project in one place. It has two main parts:

1. Train a 128x64 WGAN-GP model on real Convertible vehicle images from `Convertible/`.
2. Use the generated Convertible images in a downstream classification experiment.

The classifier section compares two setups:

- Baseline: 100 real Convertible images, no synthetic data.
- Augmented: 100 real Convertible images plus 200 randomly selected synthetic Convertible images.

Run the notebook in order: GAN setup and training first, synthetic image export next, then the classification and evaluation cells.


## 1. GAN Training and Synthetic Export

This section reads the real images from `Convertible/` and writes all GAN outputs to `convertible_outputs_wgan_gp_128x64/`.


In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset


In [ ]:
# Configuration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_dir = Path("Convertible")
output_dir = Path("convertible_outputs_wgan_gp_128x64")
sample_dir = output_dir / "samples"
checkpoint_dir = output_dir / "checkpoints"
synthetic_export_dir = output_dir / "synthetic_epoch_500"

image_width = 128
image_height = 64
channels = 3
latent_dim = 128
batch_size = 32
num_epochs = 500
lr_g = 0.0001
lr_d = 0.0001
beta1 = 0.0
beta2 = 0.9
feature_maps_g = 64
feature_maps_d = 64
n_critic = 3
gp_lambda = 10.0
synthetic_export_count = 500  # Change this before running the final export cell if needed.
synthetic_export_batch_size = 50
num_workers = 0  # Safer default for Windows/Jupyter environments
sample_interval = 10
seed = 42

random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

sample_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir.mkdir(parents=True, exist_ok=True)
synthetic_export_dir.mkdir(parents=True, exist_ok=True)

print("Device:", device)


In [ ]:
# Data

def center_crop_to_aspect(image, target_width, target_height):
    target_ratio = target_width / target_height
    width, height = image.size
    current_ratio = width / height

    if current_ratio > target_ratio:
        new_width = int(height * target_ratio)
        left = (width - new_width) // 2
        image = image.crop((left, 0, left + new_width, height))
    else:
        new_height = int(width / target_ratio)
        top = (height - new_height) // 2
        image = image.crop((0, top, width, top + new_height))

    return image


def preprocess_image(image, image_width, image_height, augment=True):
    # Vehicle photos are mostly horizontal, so a 2:1 frame keeps the car shape more natural than square padding.
    image = center_crop_to_aspect(image, image_width, image_height)
    image = image.resize((image_width, image_height), Image.Resampling.BICUBIC)
    if augment and random.random() < 0.5:
        image = ImageOps.mirror(image)

    image_np = np.asarray(image, dtype=np.float32) / 255.0
    image_tensor = torch.from_numpy(image_np).permute(2, 0, 1)
    image_tensor = (image_tensor - 0.5) / 0.5
    return image_tensor


def make_image_grid(images, nrow=4, padding=2, normalize=True, value_range=(-1, 1)):
    images = images.detach().cpu()
    if normalize:
        min_value, max_value = value_range
        images = (images - min_value) / (max_value - min_value)
    images = images.clamp(0, 1)

    batch_size, channels, height, width = images.shape
    nrow = min(nrow, batch_size)
    ncol = (batch_size + nrow - 1) // nrow

    grid = torch.ones(
        channels,
        ncol * height + padding * (ncol - 1),
        nrow * width + padding * (nrow - 1),
    )

    for idx, image in enumerate(images):
        row = idx // nrow
        col = idx % nrow
        top = row * (height + padding)
        left = col * (width + padding)
        grid[:, top:top + height, left:left + width] = image

    return grid


def save_tensor_image(image_tensor, output_path):
    image_tensor = image_tensor.detach().cpu().clamp(0, 1)
    image_np = (image_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    Image.fromarray(image_np).save(output_path)

class ConvertibleDataset(Dataset):
    def __init__(self, root_dir: Path, image_width: int, image_height: int, augment: bool = True):
        self.root_dir = Path(root_dir)
        self.image_width = image_width
        self.image_height = image_height
        self.augment = augment
        extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        self.image_paths = sorted(
            path for path in self.root_dir.iterdir()
            if path.suffix.lower() in extensions
        )
        if not self.image_paths:
            raise FileNotFoundError(f"No image files found in {self.root_dir.resolve()}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        return preprocess_image(image, self.image_width, self.image_height, augment=self.augment)


train_dataset = ConvertibleDataset(data_dir, image_width=image_width, image_height=image_height, augment=True)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=device.type == "cuda",
    drop_last=True,
)

print("Image count:", len(train_dataset))


In [ ]:
# Quick preview

sample_batch = next(iter(train_loader))
grid = make_image_grid(sample_batch[:16], nrow=4, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).cpu())
plt.title("Convertible dataset samples")
plt.axis("off")
plt.show()


In [ ]:
# Models

def weights_init(module):
    classname = module.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(module.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(module.weight.data, 1.0, 0.02)
        nn.init.constant_(module.bias.data, 0)


class Generator(nn.Module):
    def __init__(self, latent_dim, channels, feature_maps):
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, feature_maps * 8, (4, 8), 1, 0, bias=False),
            nn.BatchNorm2d(feature_maps * 8),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_maps * 8, feature_maps * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps * 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_maps * 4, feature_maps * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps * 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_maps * 2, feature_maps, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_maps, channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.model(z)


class Discriminator(nn.Module):
    def __init__(self, channels, feature_maps):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(channels, feature_maps, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_maps, feature_maps * 2, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_maps * 2, feature_maps * 4, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_maps * 4, feature_maps * 8, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_maps * 8, 1, (4, 8), 1, 0, bias=False),
        )

    def forward(self, x):
        return self.model(x).view(-1, 1)


In [ ]:
# Model initialisation

generator = Generator(latent_dim, channels, feature_maps_g).to(device)
discriminator = Discriminator(channels, feature_maps_d).to(device)

generator.apply(weights_init)
discriminator.apply(weights_init)

optimizer_G = optim.Adam(generator.parameters(), lr=lr_g, betas=(beta1, beta2))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr_d, betas=(beta1, beta2))

fixed_noise = torch.randn(16, latent_dim, 1, 1, device=device)


In [ ]:
# Helpers

def denormalize(x):
    return x * 0.5 + 0.5


def compute_gradient_penalty(critic, real_images, fake_images):
    batch_size = real_images.size(0)
    epsilon = torch.rand(batch_size, 1, 1, 1, device=real_images.device)
    interpolated = epsilon * real_images + (1 - epsilon) * fake_images
    interpolated.requires_grad_(True)

    interpolated_scores = critic(interpolated)
    gradients = torch.autograd.grad(
        outputs=interpolated_scores,
        inputs=interpolated,
        grad_outputs=torch.ones_like(interpolated_scores),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    gradients = gradients.view(batch_size, -1)
    return ((gradients.norm(2, dim=1) - 1) ** 2).mean()


def show_generated_images(epoch, generator, noise):
    generator.eval()
    with torch.no_grad():
        fake_images = generator(noise).cpu()

    grid = make_image_grid(fake_images, nrow=4, normalize=True, value_range=(-1, 1))
    plt.figure(figsize=(8, 8))
    plt.imshow(grid.permute(1, 2, 0))
    plt.title(f"Generated Convertible Images at Epoch {epoch}")
    plt.axis("off")
    plt.show()

    save_tensor_image(denormalize(grid), sample_dir / f"epoch_{epoch:03d}.png")
    generator.train()


def save_checkpoint(epoch):
    torch.save(
        {
            "epoch": epoch,
            "generator_state_dict": generator.state_dict(),
            "critic_state_dict": discriminator.state_dict(),
            "optimizer_G_state_dict": optimizer_G.state_dict(),
            "optimizer_D_state_dict": optimizer_D.state_dict(),
        },
        checkpoint_dir / f"convertible_wgan_gp_epoch_{epoch:03d}.pt",
    )


In [ ]:
# Training loop

g_losses = []
d_losses = []
gradient_penalties = []
critic_real_scores = []
critic_fake_scores = []
wasserstein_estimates = []

for epoch in range(num_epochs):
    g_loss_epoch = 0.0
    d_loss_epoch = 0.0
    gp_epoch = 0.0
    critic_real_epoch = 0.0
    critic_fake_epoch = 0.0

    for real_images in train_loader:
        real_images = real_images.to(device)
        current_batch_size = real_images.size(0)

        critic_loss_step = 0.0
        gp_step = 0.0
        critic_real_step = 0.0
        critic_fake_step = 0.0

        # Train critic
        for _ in range(n_critic):
            optimizer_D.zero_grad(set_to_none=True)

            noise = torch.randn(current_batch_size, latent_dim, 1, 1, device=device)
            fake_images = generator(noise).detach()

            real_scores = discriminator(real_images)
            fake_scores = discriminator(fake_images)
            gradient_penalty = compute_gradient_penalty(discriminator, real_images, fake_images)

            d_loss = fake_scores.mean() - real_scores.mean() + gp_lambda * gradient_penalty
            d_loss.backward()
            optimizer_D.step()

            critic_loss_step += d_loss.item()
            gp_step += gradient_penalty.item()
            critic_real_step += real_scores.mean().item()
            critic_fake_step += fake_scores.mean().item()

        # Train generator
        optimizer_G.zero_grad(set_to_none=True)

        noise = torch.randn(current_batch_size, latent_dim, 1, 1, device=device)
        fake_images = generator(noise)
        fake_scores_for_g = discriminator(fake_images)
        g_loss = -fake_scores_for_g.mean()

        g_loss.backward()
        optimizer_G.step()

        g_loss_epoch += g_loss.item()
        d_loss_epoch += critic_loss_step / n_critic
        gp_epoch += gp_step / n_critic
        critic_real_epoch += critic_real_step / n_critic
        critic_fake_epoch += critic_fake_step / n_critic

    avg_g = g_loss_epoch / len(train_loader)
    avg_d = d_loss_epoch / len(train_loader)
    avg_gp = gp_epoch / len(train_loader)
    avg_critic_real = critic_real_epoch / len(train_loader)
    avg_critic_fake = critic_fake_epoch / len(train_loader)
    g_losses.append(avg_g)
    d_losses.append(avg_d)
    gradient_penalties.append(avg_gp)
    critic_real_scores.append(avg_critic_real)
    critic_fake_scores.append(avg_critic_fake)
    wasserstein_estimates.append(avg_critic_real - avg_critic_fake)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Critic Loss: {avg_d:.4f} | G Loss: {avg_g:.4f} | "
        f"GP: {avg_gp:.4f} | C(real): {avg_critic_real:.4f} | C(fake): {avg_critic_fake:.4f}"
    )

    if (epoch + 1) % sample_interval == 0 or epoch == 0:
        show_generated_images(epoch + 1, generator, fixed_noise)
        save_checkpoint(epoch + 1)


## Gradient Curves and Convergence Analysis

This section checks whether the WGAN-GP training stayed under control. It plots generator loss, critic loss, gradient penalty, critic scores for real and fake images, and the Wasserstein distance estimate. GAN losses should not be read like normal classifier losses; the important signs are bounded curves, a controlled gradient penalty, and steadily improving samples.


In [ ]:
# Training curves and convergence analysis

if not g_losses:
    raise RuntimeError("Run the training loop before plotting convergence analysis.")

curve_output_path = output_dir / "training_curves_convergence.png"
epoch_axis = np.arange(1, len(g_losses) + 1)


def moving_average(values, window=10):
    values = np.asarray(values, dtype=np.float64)
    if len(values) < window:
        return values
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="valid")


def tail_mean(values, fraction=0.2):
    values = np.asarray(values, dtype=np.float64)
    tail_size = max(1, int(len(values) * fraction))
    return float(values[-tail_size:].mean())


def relative_tail_change(values, fraction=0.2):
    values = np.asarray(values, dtype=np.float64)
    tail_size = max(2, int(len(values) * fraction))
    tail = values[-tail_size:]
    start = float(tail[0])
    end = float(tail[-1])
    denominator = max(abs(start), 1e-8)
    return (end - start) / denominator


fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].plot(epoch_axis, g_losses, alpha=0.45, label="Generator raw")
axes[0, 0].plot(epoch_axis, d_losses, alpha=0.45, label="Critic raw")
ma_g = moving_average(g_losses)
ma_d = moving_average(d_losses)
if len(ma_g) != len(g_losses):
    ma_axis = np.arange(len(g_losses) - len(ma_g) + 1, len(g_losses) + 1)
    axes[0, 0].plot(ma_axis, ma_g, linewidth=2, label="Generator MA(10)")
    axes[0, 0].plot(ma_axis, ma_d, linewidth=2, label="Critic MA(10)")
axes[0, 0].set_title("Generator and critic loss curves")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].plot(epoch_axis, gradient_penalties, color="tab:green", alpha=0.55, label="Gradient penalty raw")
ma_gp = moving_average(gradient_penalties)
if len(ma_gp) != len(gradient_penalties):
    ma_axis = np.arange(len(gradient_penalties) - len(ma_gp) + 1, len(gradient_penalties) + 1)
    axes[0, 1].plot(ma_axis, ma_gp, color="darkgreen", linewidth=2, label="Gradient penalty MA(10)")
axes[0, 1].set_title("Gradient penalty curve")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Penalty")
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

axes[1, 0].plot(epoch_axis, critic_real_scores, label="C(real)")
axes[1, 0].plot(epoch_axis, critic_fake_scores, label="C(fake)")
axes[1, 0].set_title("Critic scores")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Score")
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

axes[1, 1].plot(epoch_axis, wasserstein_estimates, color="tab:purple", label="C(real) - C(fake)")
ma_w = moving_average(wasserstein_estimates)
if len(ma_w) != len(wasserstein_estimates):
    ma_axis = np.arange(len(wasserstein_estimates) - len(ma_w) + 1, len(wasserstein_estimates) + 1)
    axes[1, 1].plot(ma_axis, ma_w, color="indigo", linewidth=2, label="W estimate MA(10)")
axes[1, 1].set_title("Wasserstein distance estimate")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("C(real) - C(fake)")
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(curve_output_path, dpi=160, bbox_inches="tight")
plt.show()

last_fraction = 0.2
summary = {
    "epochs": len(g_losses),
    "final_generator_loss": float(g_losses[-1]),
    "final_critic_loss": float(d_losses[-1]),
    "final_gradient_penalty": float(gradient_penalties[-1]),
    "final_critic_real": float(critic_real_scores[-1]),
    "final_critic_fake": float(critic_fake_scores[-1]),
    "final_wasserstein_estimate": float(wasserstein_estimates[-1]),
    "last_20pct_generator_loss_mean": tail_mean(g_losses, last_fraction),
    "last_20pct_critic_loss_mean": tail_mean(d_losses, last_fraction),
    "last_20pct_gradient_penalty_mean": tail_mean(gradient_penalties, last_fraction),
    "last_20pct_wasserstein_estimate_mean": tail_mean(wasserstein_estimates, last_fraction),
    "last_20pct_generator_loss_relative_change": relative_tail_change(g_losses, last_fraction),
    "last_20pct_gradient_penalty_relative_change": relative_tail_change(gradient_penalties, last_fraction),
    "last_20pct_wasserstein_relative_change": relative_tail_change(wasserstein_estimates, last_fraction),
}

print("Convergence summary")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("\nInterpretation guide")
print("- Generator/critic losses in WGAN-GP do not need to monotonically decrease like classifier loss.")
print("- A useful sign is that curves stop exploding and fluctuate around a bounded range.")
print("- Gradient penalty should stay controlled; very large spikes suggest critic instability.")
print("- C(real) should generally be higher than C(fake); their gap is the Wasserstein estimate.")
print("- Final judgment should combine these curves with visual sample quality and FID/IS results.")
print(f"\nSaved curve figure to: {curve_output_path}")


In [ ]:
# Final generation

generator.eval()
with torch.no_grad():
    z = torch.randn(25, latent_dim, 1, 1, device=device)
    generated = generator(z).cpu()

grid = make_image_grid(generated, nrow=5, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0))
plt.title("Final generated Convertible images")
plt.axis("off")
plt.show()

save_tensor_image(denormalize(make_image_grid(generated, nrow=5, normalize=True, value_range=(-1, 1))), output_dir / "final_generated_grid.png")


In [ ]:
# Export individual synthetic Convertible images

def export_synthetic_images(generator_model, output_dir, image_count, batch_size):
    output_dir.mkdir(parents=True, exist_ok=True)
    generator_model.eval()

    exported = 0
    with torch.no_grad():
        while exported < image_count:
            current_batch_size = min(batch_size, image_count - exported)
            noise = torch.randn(current_batch_size, latent_dim, 1, 1, device=device)
            fake_images = generator_model(noise).cpu()

            for image in fake_images:
                exported += 1
                output_path = output_dir / f"synthetic_convertible_{exported:03d}.png"
                save_tensor_image(denormalize(image), output_path)

    print(f"Exported {exported} synthetic images to {output_dir}")


export_synthetic_images(
    generator,
    synthetic_export_dir,
    image_count=synthetic_export_count,
    batch_size=synthetic_export_batch_size,
)


## GAN Evaluation Metrics: FID and Inception Score

This section adds two quantitative checks for the generated Convertible images.

- **FID (Frechet Inception Distance):** compares the real and generated image distributions in Inception feature space. Lower is better.
- **IS (Inception Score):** estimates whether generated images are both recognizable and diverse according to an Inception model. Higher is usually better.

These metrics use pretrained InceptionV3 weights. The first run may need internet access, unless the weights are already cached. The generated images are 128x64, but InceptionV3 expects 299x299 inputs, so the metric images are resized before evaluation. Because of that, the scores should be treated as useful indicators, not as the only final judgment.


In [ ]:
# FID and Inception Score analysis

from math import ceil

try:
    from scipy import linalg
except Exception:
    linalg = None

from torchvision import models, transforms

metric_real_dir = data_dir
metric_fake_dir = synthetic_export_dir
metric_max_real_images = 500
metric_max_fake_images = 500
metric_batch_size = 32
metric_splits = 10
metric_seed = seed


class MetricImageDataset(Dataset):
    def __init__(self, root_dir, transform, max_images=None, seed=42):
        self.root_dir = Path(root_dir)
        self.transform = transform
        extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        self.image_paths = sorted(
            path for path in self.root_dir.iterdir()
            if path.is_file() and path.suffix.lower() in extensions
        )
        if not self.image_paths:
            raise FileNotFoundError(f"No image files found in {self.root_dir.resolve()}")
        if max_images is not None and len(self.image_paths) > max_images:
            rng = random.Random(seed)
            self.image_paths = sorted(rng.sample(self.image_paths, max_images))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        return self.transform(image)


metric_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])


def load_inception_model(device):
    weights = models.Inception_V3_Weights.DEFAULT
    model = models.inception_v3(weights=weights, aux_logits=True)
    model.to(device)
    model.eval()
    return model


def collect_inception_features_and_probs(image_dir, max_images=None):
    dataset = MetricImageDataset(
        image_dir,
        transform=metric_transform,
        max_images=max_images,
        seed=metric_seed,
    )
    loader = DataLoader(
        dataset,
        batch_size=metric_batch_size,
        shuffle=False,
        num_workers=0,
    )

    feature_batches = []
    prob_batches = []
    captured_features = []

    def capture_avgpool_features(_module, _inputs, output):
        captured_features.append(output.flatten(1).detach().cpu())

    hook = inception_model.avgpool.register_forward_hook(capture_avgpool_features)
    try:
        with torch.no_grad():
            for images in loader:
                images = images.to(device)
                captured_features.clear()
                logits = inception_model(images)
                if isinstance(logits, tuple):
                    logits = logits[0]

                features = captured_features.pop()
                probs = torch.softmax(logits, dim=1).detach().cpu()

                feature_batches.append(features)
                prob_batches.append(probs)
    finally:
        hook.remove()

    features = torch.cat(feature_batches, dim=0).numpy().astype(np.float64)
    probs = torch.cat(prob_batches, dim=0).numpy().astype(np.float64)
    return features, probs, len(dataset)


def calculate_fid(real_features, fake_features, eps=1e-6):
    mu_real = real_features.mean(axis=0)
    mu_fake = fake_features.mean(axis=0)
    sigma_real = np.cov(real_features, rowvar=False)
    sigma_fake = np.cov(fake_features, rowvar=False)

    diff = mu_real - mu_fake
    cov_product = sigma_real.dot(sigma_fake)

    if linalg is not None:
        covmean, _ = linalg.sqrtm(cov_product, disp=False)
        if not np.isfinite(covmean).all():
            offset = np.eye(sigma_real.shape[0]) * eps
            covmean = linalg.sqrtm((sigma_real + offset).dot(sigma_fake + offset))
        if np.iscomplexobj(covmean):
            covmean = covmean.real
        trace_covmean = np.trace(covmean)
    else:
        # Fallback if scipy is unavailable. This is less numerically robust than scipy.linalg.sqrtm.
        eigenvalues = np.linalg.eigvals(cov_product)
        trace_covmean = np.sum(np.sqrt(np.maximum(eigenvalues.real, 0.0)))

    fid = diff.dot(diff) + np.trace(sigma_real) + np.trace(sigma_fake) - 2.0 * trace_covmean
    return float(np.real(fid))


def calculate_inception_score(probs, splits=10, eps=1e-16):
    splits = max(1, min(splits, len(probs)))
    split_scores = []
    for part in np.array_split(probs, splits):
        marginal = np.mean(part, axis=0, keepdims=True)
        kl_divergence = part * (np.log(part + eps) - np.log(marginal + eps))
        split_scores.append(np.exp(np.mean(np.sum(kl_divergence, axis=1))))
    return float(np.mean(split_scores)), float(np.std(split_scores))


try:
    inception_model = load_inception_model(device)

    real_features, _, real_metric_count = collect_inception_features_and_probs(
        metric_real_dir,
        max_images=metric_max_real_images,
    )
    fake_features, fake_probs, fake_metric_count = collect_inception_features_and_probs(
        metric_fake_dir,
        max_images=metric_max_fake_images,
    )

    fid_score = calculate_fid(real_features, fake_features)
    is_mean, is_std = calculate_inception_score(fake_probs, splits=metric_splits)

    print("GAN evaluation metrics")
    print("Real images used for FID:", real_metric_count)
    print("Synthetic images used for FID/IS:", fake_metric_count)
    print(f"FID: {fid_score:.4f}  (lower is better)")
    print(f"Inception Score: {is_mean:.4f} +/- {is_std:.4f}  (higher is usually better)")
except Exception as exc:
    print("Could not compute FID / Inception Score.")
    print("Reason:", exc)
    print("Check that synthetic images exist and pretrained InceptionV3 weights are available.")


## Notes

- Training time depends heavily on the machine. CPU training will be slow.
- If the run is too slow, reduce `num_epochs`, `batch_size`, or the image size for a quick test.
- Change `synthetic_export_count` before the export cell if you need a different number of generated images.
- For better image quality, the next reasonable experiments would be DiffAugment, ADA, or StyleGAN2-ADA.


## 2. Classification with Synthetic Convertible Data

This section takes 200 random synthetic images from the GAN export folder and compares them against a baseline that uses only 100 real Convertible images.


In [ ]:
# If kagglehub is missing in a fresh Colab/runtime, uncomment this line:
# !pip install kagglehub

from pathlib import Path
import random

import kagglehub
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms


In [ ]:
# Configuration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

project_root = Path.cwd()
synthetic_dir = project_root / "convertible_outputs_wgan_gp_128x64" / "synthetic_epoch_500"

image_height = 64
image_width = 128
batch_size = 64
num_epochs = 10
learning_rate = 3e-4
seed = 42

target_class_candidates = ["Convertible", "Convertibles"]
target_real_keep = 100
synthetic_keep = 200
expected_synthetic_count = None
synthetic_loss_weight = 1.0

random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print("Device:", device)
print("Synthetic directory:", synthetic_dir)


In [ ]:
# Download and load the real classification dataset

dataset_path = Path(kagglehub.dataset_download("ademboukhris/cars-body-type-cropped"))
cars_root = dataset_path / "Cars_Body_Type"

# ResNet pretrained weights expect ImageNet normalization. Use the same transform for real and synthetic images.
transform = transforms.Compose([
    transforms.Resize((image_height, image_width)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

preview_transform = transforms.Compose([
    transforms.Resize((image_height, image_width)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(root=cars_root / "train", transform=transform)
test_dataset = datasets.ImageFolder(root=cars_root / "test", transform=transform)

print("Classes:", train_dataset.classes)
print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))


In [ ]:
# Find the Convertible class label robustly

def find_target_class(classes, candidates):
    normalized = {name.lower().replace("-", "").replace("_", "").replace(" ", ""): name for name in classes}
    for candidate in candidates:
        key = candidate.lower().replace("-", "").replace("_", "").replace(" ", "")
        if key in normalized:
            return normalized[key]
    raise ValueError(f"Could not find a Convertible-like class. Available classes: {classes}")


target_class_name = find_target_class(train_dataset.classes, target_class_candidates)
target_label = train_dataset.class_to_idx[target_class_name]

print("Target class:", target_class_name)
print("Target label:", target_label)


In [ ]:
# Create an imbalanced training set by reducing real Convertible samples

target_indices = []
other_indices = []

for idx, (_, label) in enumerate(train_dataset.samples):
    if label == target_label:
        target_indices.append(idx)
    else:
        other_indices.append(idx)

keep_count = min(target_real_keep, len(target_indices))
reduced_target_indices = random.sample(target_indices, keep_count)
imbalanced_indices = reduced_target_indices + other_indices
random.shuffle(imbalanced_indices)

imbalanced_dataset = Subset(train_dataset, imbalanced_indices)

print("Target original train count:", len(target_indices))
print("Target kept in imbalanced train count:", len(reduced_target_indices))
print("Other train count:", len(other_indices))
print("Full real train size:", len(train_dataset))
print("Imbalanced train size:", len(imbalanced_dataset))


In [ ]:
# Synthetic Convertible dataset generated by our WGAN-GP model

class SyntheticConvertibleDataset(Dataset):
    def __init__(self, root_dir, transform, label, max_count=None, seed=42):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.label = label
        self.image_paths = sorted(
            path for path in self.root_dir.iterdir()
            if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        )
        if not self.image_paths:
            raise FileNotFoundError(f"No synthetic images found in {self.root_dir.resolve()}")
        if max_count is not None and len(self.image_paths) > max_count:
            rng = random.Random(seed)
            self.image_paths = sorted(rng.sample(self.image_paths, max_count))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        return self.transform(image), self.label


synthetic_dataset = SyntheticConvertibleDataset(
    synthetic_dir,
    transform=transform,
    label=target_label,
    max_count=synthetic_keep,
    seed=seed,
)
synthetic_preview_dataset = SyntheticConvertibleDataset(
    synthetic_dir,
    transform=preview_transform,
    label=target_label,
    max_count=synthetic_keep,
    seed=seed,
)

print("Synthetic Convertible count:", len(synthetic_dataset))
print("Synthetic random keep:", synthetic_keep)
print("Synthetic label:", target_label, target_class_name)

if expected_synthetic_count is not None and len(synthetic_dataset) != expected_synthetic_count:
    print(f"Warning: expected {expected_synthetic_count} synthetic images, found {len(synthetic_dataset)}")


In [ ]:
# Dataset wrappers

class WeightedDataset(Dataset):
    def __init__(self, dataset, sample_weight):
        self.dataset = dataset
        self.sample_weight = float(sample_weight)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        return image, label, self.sample_weight


imbalanced_weighted = WeightedDataset(imbalanced_dataset, sample_weight=1.0)
synthetic_weighted = WeightedDataset(synthetic_dataset, sample_weight=synthetic_loss_weight)

augmented_dataset = ConcatDataset([imbalanced_weighted, synthetic_weighted])

imbalanced_train_loader = DataLoader(
    imbalanced_weighted,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

augmented_train_loader = DataLoader(
    augmented_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

print("Imbalanced train size:", len(imbalanced_weighted))
print("Augmented train size:", len(augmented_dataset))
print("Imbalanced target count:", len(reduced_target_indices))
print("Augmented target count:", len(reduced_target_indices) + len(synthetic_dataset))
print("Synthetic loss weight:", synthetic_loss_weight)


In [ ]:
# Classifier: ResNet18 with pretrained fallback

def make_model():
    num_classes = len(train_dataset.classes)
    try:
        weights = models.ResNet18_Weights.DEFAULT
        model = models.resnet18(weights=weights)
        print("Using pretrained ResNet18 weights.")
    except Exception as exc:
        print("Could not load pretrained weights; using randomly initialized ResNet18.")
        print("Reason:", exc)
        model = models.resnet18(weights=None)

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    # Fine-tune the last block and classifier. This is more stable than training all layers on a small dataset.
    for name, parameter in model.named_parameters():
        parameter.requires_grad = name.startswith("layer4") or name.startswith("fc")

    return model.to(device)


In [ ]:
# Training and evaluation helpers

def train_model(model, train_loader, epochs, title):
    criterion = nn.CrossEntropyLoss(reduction="none")
    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=learning_rate,
        weight_decay=1e-4,
    )

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for images, labels, sample_weights in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            sample_weights = sample_weights.to(device).float()

            outputs = model(images)
            per_sample_loss = criterion(outputs, labels)
            loss = (per_sample_loss * sample_weights).sum() / sample_weights.sum().clamp_min(1.0)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        avg_loss = total_loss / total
        accuracy = correct / total
        print(f"{title} | Epoch [{epoch + 1}/{epochs}] | Loss: {avg_loss:.4f} | Train Acc: {accuracy:.4f}")

    return model


def evaluate_model(model, loader, title):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu()

            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    print(classification_report(all_labels, all_preds, target_names=train_dataset.classes, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return all_labels, all_preds


In [ ]:
# Case 1: baseline, only 100 real Convertible samples

imbalanced_model = make_model()
imbalanced_model = train_model(
    imbalanced_model,
    imbalanced_train_loader,
    epochs=num_epochs,
    title="Baseline",
)

imbalanced_labels, imbalanced_preds = evaluate_model(
    imbalanced_model,
    test_loader,
    title="Baseline evaluation with 100 real Convertible images and no synthetic data",
)


In [ ]:
# Case 2: equal-weight augmented, 100 real Convertible + 200 random synthetic Convertible images

augmented_model = make_model()
augmented_model = train_model(
    augmented_model,
    augmented_train_loader,
    epochs=num_epochs,
    title="Equal-weight augmented",
)

augmented_labels, augmented_preds = evaluate_model(
    augmented_model,
    test_loader,
    title="Equal-weight augmented evaluation with synthetic Convertible images",
)


In [ ]:
# Inspect a few synthetic samples used in training

preview_count = min(8, len(synthetic_preview_dataset))
plt.figure(figsize=(14, 4))

for i in range(preview_count):
    image_tensor, label = synthetic_preview_dataset[i]
    image = image_tensor.clamp(0, 1).permute(1, 2, 0)

    plt.subplot(2, 4, i + 1)
    plt.imshow(image)
    plt.title(train_dataset.classes[label])
    plt.axis("off")

plt.tight_layout()
plt.show()


## Interpretation

Use the two reports in this order:

- Baseline tells us how the classifier performs when Convertible is reduced to 100 real images.
- Equal-weight augmented tells us whether 200 randomly selected synthetic Convertible images recover some of that lost Convertible performance when treated the same as real images.

For the synthetic data to be useful, the Convertible `f1-score` should improve over the baseline without collapsing precision.
